<a href="https://colab.research.google.com/github/ihsann2410010447/statistika-probabilitas-uniska/blob/main/M13_Praktikum_Dua_Sampel_Chi_Square_2410010447_Akbar_Ihsan_Nugraha.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ⚖️ Modul 13 — Praktikum Uji Hipotesis Dua Sampel & Chi-Square

### Studi Kasus: Adult Census Income (48.842 orang) + Exercise (pengukuran berpasangan)

**Mata Kuliah:** Statistika & Probabilitas (TIF1501)
**Program Studi:** Teknik Informatika (55201) · FTI · Uniska MAB Banjarmasin
**Semester:** 5 · 3 SKS · CPMK072 (Sub-CPMK 072.2) · Milestone 5

---

## 🎯 Tujuan Pembelajaran

1. **Membandingkan** dua grup independen dengan Welch t-test, pooled t-test, dan Mann-Whitney U.
2. **Menganalisis** data berpasangan dengan paired t-test dan Wilcoxon signed-rank.
3. **Menguji** selisih dua proporsi, lengkap dengan CI dan relative risk.
4. **Menerapkan** chi-square independensi, goodness-of-fit, residual terstandar, dan Fisher exact.

---

## 📋 Alur Notebook

| Bagian | Isi | Waktu |
|--------|-----|-------|
| **A** | Setup & Load Data | 5 menit |
| **B** | Fungsi Effect Size | 5 menit |
| **C** | Welch vs Pooled t-test | 10 menit |
| **D** | Mann-Whitney U (data skewed) | 10 menit |
| **E** | Paired t-test vs uji independen | 15 menit |
| **F** | Uji Dua Proporsi | 10 menit |
| **G** | Chi-square Independensi + Residual | 15 menit |
| **H** | Goodness-of-Fit & Fisher Exact | 10 menit |
| **I** | Template Milestone 5 + Refleksi | 10 menit |

**Total: ± 90 menit**

---

## 💡 Dua Dataset

| Dataset | Desain | Isi |
|---|---|---|
| **Adult Census Income** (UCI) | Sampel **independen** | 48.842 orang dewasa AS (1994): usia, pendidikan, pekerjaan, jam kerja/minggu, gender, capital gain, dan penghasilan > $50K atau tidak |
| **Exercise** (seaborn) | Sampel **berpasangan** | 30 orang; denyut nadi diukur pada menit ke-1, 15, dan 30 saat istirahat, jalan kaki, atau lari |

**Catatan etis**: kita membandingkan kelompok gender karena itu variabel di data sensus. Hasilnya menggambarkan **pola pada data 1994**, bukan kemampuan individu — dan bukan bukti sebab-akibat.


---

# 🔧 A. Setup & Load Data


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)
ALPHA = 0.05

# --- Adult Census Income (UCI) ---
cols = ['age', 'workclass', 'fnlwgt', 'education', 'education_num', 'marital_status',
        'occupation', 'relationship', 'race', 'sex', 'capital_gain', 'capital_loss',
        'hours_per_week', 'native_country', 'income']
url_adult = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/adult-all.csv"
adult = pd.read_csv(url_adult, header=None, names=cols, na_values='?', skipinitialspace=True)
adult['high'] = (adult['income'] == '>50K').astype(int)   # 1 = penghasilan > $50K

# --- Exercise (pengukuran berpasangan) ---
url_ex = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/exercise.csv"
exercise = pd.read_csv(url_ex, index_col=0)

print(f"Adult   : {adult.shape[0]:,} orang × {adult.shape[1]} kolom")
print(f"Exercise: {exercise.shape[0]} baris = {exercise['id'].nunique()} orang × 3 waktu ukur")
print(f"\nProporsi penghasilan > $50K: {adult.high.mean():.1%}")
adult[['age', 'education', 'sex', 'hours_per_week', 'capital_gain', 'income']].head()


---

# 🧰 B. Fungsi Effect Size

p-value menjawab "apakah ada beda?". Effect size menjawab "**seberapa besar?**" — wajib dilaporkan, apalagi saat n sangat besar.


In [ ]:
def cohens_d(a, b):
    """Selisih mean dalam satuan pooled standard deviation."""
    a, b = np.asarray(a), np.asarray(b)
    sp = np.sqrt(((len(a)-1)*a.var(ddof=1) + (len(b)-1)*b.var(ddof=1)) / (len(a)+len(b)-2))
    return (a.mean() - b.mean()) / sp

def welch_ci(a, b, alpha=ALPHA):
    """CI selisih mean dengan derajat bebas Welch-Satterthwaite."""
    va, vb = np.var(a, ddof=1)/len(a), np.var(b, ddof=1)/len(b)
    df = (va + vb)**2 / (va**2/(len(a)-1) + vb**2/(len(b)-1))
    diff, se = np.mean(a) - np.mean(b), np.sqrt(va + vb)
    t = stats.t.ppf(1 - alpha/2, df)
    return diff, diff - t*se, diff + t*se

def cramers_v(table):
    chi2 = stats.chi2_contingency(table, correction=False)[0]
    n = np.asarray(table).sum()
    return np.sqrt(chi2 / (n * (min(np.asarray(table).shape) - 1)))

def prob_superiority(a, b):
    """P(X > Y) + 0.5·P(X = Y) — effect size untuk Mann-Whitney (0.5 = tidak ada beda)."""
    u = stats.mannwhitneyu(a, b).statistic
    return u / (len(a) * len(b))

print("Fungsi siap: cohens_d, welch_ci, cramers_v, prob_superiority")


---

# 📊 C. Dua Sampel Independen: Welch vs Pooled

**Pertanyaan**: apakah rata-rata jam kerja per minggu pria dan wanita berbeda?

- **H₀**: μ_pria = μ_wanita  **H₁**: μ_pria ≠ μ_wanita


In [ ]:
male = adult.loc[adult.sex == 'Male', 'hours_per_week']
female = adult.loc[adult.sex == 'Female', 'hours_per_week']

print(f"Pria   : mean = {male.mean():.2f} jam, sd = {male.std():.2f}, n = {len(male):,}")
print(f"Wanita : mean = {female.mean():.2f} jam, sd = {female.std():.2f}, n = {len(female):,}")

lev = stats.levene(male, female)
print(f"\nLevene (H₀: varians sama): p = {lev.pvalue:.4f}")

t_pooled = stats.ttest_ind(male, female, equal_var=True)
t_welch = stats.ttest_ind(male, female, equal_var=False)
diff, lo, hi = welch_ci(male, female)
print(f"\nPooled t-test : t = {t_pooled.statistic:.2f}, p = {t_pooled.pvalue:.2e}")
print(f"Welch t-test  : t = {t_welch.statistic:.2f}, p = {t_welch.pvalue:.2e}")
print(f"\nSelisih = {diff:.2f} jam, CI 95% [{lo:.2f}, {hi:.2f}]")
print(f"Cohen's d = {cohens_d(male, female):.2f}  (0.2 kecil · 0.5 sedang · 0.8 besar)")
print("\n💡 Levene signifikan, tetapi SD hampir sama (12.1 vs 11.9) → pooled & Welch hampir identik.")
print("   Karena Welch aman di kedua kondisi, jadikan Welch sebagai default.")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
sns.histplot(data=adult, x='hours_per_week', hue='sex', bins=40, stat='density', common_norm=False,
             palette={'Male': '#4338CA', 'Female': '#BE123C'}, ax=axes[0])
axes[0].set_title('Distribusi jam kerja per minggu', fontweight='bold')
axes[0].set_xlim(0, 80)

means = adult.groupby('sex')['hours_per_week'].agg(['mean', 'sem'])
axes[1].bar(means.index, means['mean'], yerr=1.96*means['sem'], capsize=10,
            color=['#BE123C', '#4338CA'], edgecolor='black')
axes[1].set_ylim(30, 45); axes[1].set_ylabel('jam / minggu')
axes[1].set_title('Mean ± CI 95% (CI sangat sempit karena n besar)', fontweight='bold')
plt.tight_layout(); plt.show()
print("💡 Puncak di 40 jam untuk keduanya; wanita lebih banyak di jam paruh waktu (< 40).")


### C.1 Jebakan n besar: semuanya signifikan

Dengan puluhan ribu data, hampir setiap perbandingan menghasilkan p ≈ 0. Bandingkan beberapa variabel:


In [ ]:
rows = []
for var in ['hours_per_week', 'age', 'education_num']:
    a = adult.loc[adult.high == 1, var]
    b = adult.loc[adult.high == 0, var]
    t = stats.ttest_ind(a, b, equal_var=False)
    rows.append([var, a.mean(), b.mean(), t.pvalue, cohens_d(a, b)])
tbl = pd.DataFrame(rows, columns=['variabel', 'mean > $50K', 'mean ≤ $50K', 'p-value', "Cohen's d"])
print(tbl.round(3).to_string(index=False))
print("\n💡 Semua p ≈ 0 — p-value tidak membantu memeringkat. Cohen's d yang membedakan besar efeknya.")


---

# 📉 D. Mann-Whitney U: Data Sangat Skewed

**Pertanyaan**: apakah capital gain kelompok berpenghasilan > $50K lebih besar daripada kelompok ≤ $50K?


In [ ]:
hi_cg = adult.loc[adult.high == 1, 'capital_gain']
lo_cg = adult.loc[adult.high == 0, 'capital_gain']

print(f"Proporsi capital gain = 0 : {(adult.capital_gain == 0).mean():.1%}")
print(f"Skewness capital gain      : {stats.skew(adult.capital_gain):.1f}\n")
summary = pd.DataFrame({
    'mean': [hi_cg.mean(), lo_cg.mean()],
    'median': [hi_cg.median(), lo_cg.median()],
    '% punya gain > 0': [(hi_cg > 0).mean()*100, (lo_cg > 0).mean()*100],
    'maks': [hi_cg.max(), lo_cg.max()],
}, index=['> $50K', '≤ $50K']).round(1)
print(summary)

mw = stats.mannwhitneyu(hi_cg, lo_cg, alternative='two-sided')
ps = prob_superiority(hi_cg, lo_cg)
print(f"\nMann-Whitney U = {mw.statistic:,.0f}, p = {mw.pvalue:.2e}")
print(f"P(X > Y) = {ps:.2f}  → 0.50 berarti tidak ada beda")
print("\n💡 Median keduanya 0, tetapi Mann-Whitney tetap signifikan: ia membandingkan SELURUH distribusi")
print("   (peringkat), bukan median saja. Kelompok > $50K jauh lebih sering punya capital gain > 0.")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
for label, data, color in [('> $50K', hi_cg, '#4338CA'), ('≤ $50K', lo_cg, '#BE123C')]:
    nz = np.sort(data[data > 0])
    ax.plot(nz, 1 - np.arange(1, len(nz)+1)/len(data) , lw=2.5, color=color, label=f'{label} (gain > 0: {len(nz)/len(data):.1%})')
ax.set_xscale('log'); ax.set_xlabel('capital gain ($, skala log)'); ax.set_ylabel('P(gain > x)')
ax.set_title('Kurva survival capital gain — kurva biru selalu di atas', fontweight='bold')
ax.legend(); plt.tight_layout(); plt.show()


---

# 🔗 E. Dua Sampel Berpasangan: Paired t-test

**Pertanyaan**: apakah 15 menit jalan kaki menaikkan denyut nadi? 10 orang yang **sama** diukur pada menit ke-1 dan ke-15.

- **H₀**: μ_d = 0  **H₁**: μ_d ≠ 0, dengan d = nadi menit 15 − nadi menit 1


In [ ]:
wide = exercise.pivot_table(index=['id', 'kind', 'diet'], columns='time', values='pulse').reset_index()
walk = wide[wide.kind == 'walking'][['id', 'diet', '1 min', '15 min']].copy()
walk['d'] = walk['15 min'] - walk['1 min']
print(walk.to_string(index=False))

before, after = walk['1 min'], walk['15 min']
paired = stats.ttest_rel(after, before)
indep = stats.ttest_ind(after, before, equal_var=False)
ci_d = stats.t.interval(1 - ALPHA, len(walk) - 1, loc=walk.d.mean(), scale=stats.sem(walk.d))
wil = stats.wilcoxon(after, before)

print(f"\nd̄ = {walk.d.mean():.2f} bpm, s_d = {walk.d.std():.2f}, CI 95% [{ci_d[0]:.2f}, {ci_d[1]:.2f}]")
print(f"Korelasi menit 1 vs 15: r = {np.corrcoef(before, after)[0,1]:.2f}")
print(f"\n✅ Paired t-test      : t = {paired.statistic:.2f}, p = {paired.pvalue:.4f}")
print(f"✅ Wilcoxon signed-rank: p = {wil.pvalue:.4f}")
print(f"❌ Uji independen (SALAH desain): t = {indep.statistic:.2f}, p = {indep.pvalue:.4f}")
print(f"\nSD antar-orang ≈ {before.std():.1f}–{after.std():.1f} bpm, sedangkan SD selisih hanya {walk.d.std():.1f} bpm.")
print("💡 Uji independen mencampur perbedaan ANTAR orang ke dalam 'noise' sehingga kenaikan 3.5 bpm tenggelam.")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for _, r in walk.iterrows():
    axes[0].plot([0, 1], [r['1 min'], r['15 min']], 'o-', color='#4338CA', alpha=0.7)
axes[0].set_xticks([0, 1]); axes[0].set_xticklabels(['menit 1', 'menit 15'])
axes[0].set_ylabel('denyut nadi (bpm)')
axes[0].set_title('Setiap garis = satu orang: hampir semua naik', fontweight='bold')
axes[1].bar(range(1, 11), walk.d, color=['#059669' if v > 0 else '#94A3B8' for v in walk.d], edgecolor='black')
axes[1].axhline(0, color='black'); axes[1].axhline(walk.d.mean(), color='#BE123C', ls='--', label=f'd̄ = {walk.d.mean():.1f}')
axes[1].set_xlabel('orang ke-'); axes[1].set_ylabel('selisih (bpm)')
axes[1].set_title('Selisih per orang: 9 naik, 1 tetap', fontweight='bold'); axes[1].legend()
plt.tight_layout(); plt.show()


In [ ]:
# Bandingkan ketiga aktivitas (menit 1 vs menit 30)
rows = []
for kind in ['rest', 'walking', 'running']:
    g = wide[wide.kind == kind]
    rows.append([kind, g['1 min'].mean(), g['30 min'].mean(), (g['30 min'] - g['1 min']).mean(),
                 stats.ttest_rel(g['30 min'], g['1 min']).pvalue])
print(pd.DataFrame(rows, columns=['aktivitas', 'menit 1', 'menit 30', 'd̄', 'p paired']).round(4).to_string(index=False))
print("\n💡 Lari: efek besar (+30 bpm). Istirahat: hampir tidak berubah — sesuai intuisi.")


---

# 🎯 F. Uji Dua Proporsi

**Pertanyaan**: apakah proporsi penghasilan > $50K berbeda antara pria dan wanita?

- **H₀**: p_pria = p_wanita  **H₁**: p_pria ≠ p_wanita


In [ ]:
grp = adult.groupby('sex')['high'].agg(['sum', 'count'])
grp['proporsi'] = grp['sum'] / grp['count']
print(grp)

k = grp.loc[['Male', 'Female'], 'sum'].values
n = grp.loc[['Male', 'Female'], 'count'].values
z, p_prop = proportions_ztest(k, n)
pm, pf = k / n
se = np.sqrt(pm*(1-pm)/n[0] + pf*(1-pf)/n[1])
print(f"\nSelisih = {(pm-pf)*100:.1f} poin, CI 95% [{(pm-pf-1.96*se)*100:.1f}, {(pm-pf+1.96*se)*100:.1f}]")
print(f"z = {z:.2f}, p = {p_prop:.2e}")
print(f"Relative risk = {pm/pf:.2f} → proporsi pria berpenghasilan > $50K {pm/pf:.1f}× lipat wanita")

chi_equiv = stats.chi2_contingency(pd.crosstab(adult.sex, adult.income), correction=False)
print(f"\nSetara chi-square 2×2: χ² = {chi_equiv[0]:.1f} = z² = {z**2:.1f} ✅")


### F.1 Asosiasi ≠ sebab-akibat

Apakah selisih ini disebabkan gender? Data observasional tidak bisa menjawabnya. Lihat bagaimana selisih berubah setelah membandingkan orang dengan pendidikan **dan** jam kerja serupa:


In [ ]:
adult['full_time'] = np.where(adult.hours_per_week >= 40, '≥ 40 jam', '< 40 jam')
strata = (adult.groupby(['full_time', 'education', 'sex'])['high'].mean().unstack('sex'))
strata = strata[adult.groupby(['full_time', 'education']).size() >= 300]
strata['selisih (poin)'] = (strata['Male'] - strata['Female']) * 100
print((strata * [100, 100, 1]).round(1).loc[(slice(None), ['HS-grad', 'Some-college', 'Bachelors', 'Masters']), :]
      .rename(columns={'Female': '% wanita', 'Male': '% pria'}))
print(f"\nSelisih kasar (tanpa stratifikasi) = {(pm-pf)*100:.1f} poin")
print("💡 Selisih tetap ada dalam strata yang sama, tetapi besarnya berubah — variabel lain ikut berperan.")
print("   Variabel seperti status pernikahan, pekerjaan, dan usia juga bisa menjadi confounder.")
print("   Kesimpulan yang aman: ada ASOSIASI; klaim sebab-akibat butuh desain/analisis lain.")


---

# 🧮 G. Chi-Square Independensi: Pendidikan × Penghasilan

- **H₀**: tingkat pendidikan independen dari kategori penghasilan
- **H₁**: ada hubungan

16 kategori pendidikan digabung menjadi 5 kelompok yang bermakna.


In [ ]:
def edu_group(e):
    if e in ['Preschool', '1st-4th', '5th-6th', '7th-8th', '9th', '10th', '11th', '12th']:
        return '1. Tidak lulus SMA'
    if e == 'HS-grad':
        return '2. Lulus SMA'
    if e in ['Some-college', 'Assoc-voc', 'Assoc-acdm']:
        return '3. Diploma / some college'
    if e == 'Bachelors':
        return '4. Sarjana (S1)'
    return '5. Pascasarjana'

adult['edu'] = adult['education'].map(edu_group)
ct = pd.crosstab(adult['edu'], adult['income'])
chi2, p_chi, dof, expected = stats.chi2_contingency(ct)
resid = (ct - expected) / np.sqrt(expected)

print("Tabel kontingensi (observed):"); print(ct)
print(f"\nExpected minimum = {expected.min():.0f} (≥ 5 ✅)")
print(f"χ² = {chi2:,.1f}, df = {dof}, p = {p_chi:.2e}")
print(f"Cramér's V = {cramers_v(ct):.3f}  (0.1 lemah · 0.3 sedang · 0.5 kuat)")
print("\n% berpenghasilan > $50K per kelompok:")
print((pd.crosstab(adult['edu'], adult['income'], normalize='index')['>50K'] * 100).round(1))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
pct = pd.crosstab(adult['edu'], adult['income'], normalize='index') * 100
pct.plot(kind='barh', stacked=True, ax=axes[0], color=['#94A3B8', '#4338CA'], edgecolor='black')
axes[0].set_xlabel('%'); axes[0].set_title('Komposisi penghasilan per pendidikan', fontweight='bold')
axes[0].invert_yaxis(); axes[0].legend(title='income', loc='lower right')
sns.heatmap(resid, annot=True, fmt='.1f', cmap='RdBu_r', center=0, ax=axes[1], cbar_kws={'label': 'residual terstandar'})
axes[1].set_title('Residual terstandar (O − E)/√E — |r| > 2 menonjol', fontweight='bold')
plt.tight_layout(); plt.show()
print("💡 Biru = lebih banyak dari harapan, merah = lebih sedikit. Pascasarjana & S1 'kelebihan' > $50K.")


---

# 📐 H. Goodness-of-Fit & Fisher Exact

## H.1 Goodness-of-Fit: apakah sampel representatif?

Populasi (seluruh dataset) punya komposisi gender tertentu. Kita uji dua cara mengambil sampel 500 orang.


In [ ]:
pop_prop = adult['sex'].value_counts(normalize=True)
print("Proporsi populasi:"); print(pop_prop.round(4))

samples = {
    'Sampel acak': adult.sample(500, random_state=1),
    'Sampel dari yang kerja ≥ 50 jam': adult[adult.hours_per_week >= 50].sample(500, random_state=1),
}
for name, smp in samples.items():
    obs = smp['sex'].value_counts().reindex(pop_prop.index)
    exp = pop_prop.values * 500
    chi2_gof, p_gof = stats.chisquare(obs.values, exp)
    verdict = 'representatif ✅' if p_gof >= ALPHA else 'BIAS ❌'
    print(f"\n{name}")
    print(f"  observed : Male = {int(obs['Male'])}, Female = {int(obs['Female'])}")
    print(f"  expected : Male = {exp[0]:.1f}, Female = {exp[1]:.1f}")
    print(f"  χ² = {chi2_gof:.2f}, p = {p_gof:.2e} → {verdict}")
print("\n💡 Cara mengambil sampel menentukan siapa yang terwakili (ingat Modul 1: sampling bias).")


## H.2 Fisher Exact: saat sel tabel terlalu kecil

Sampel 30 pekerja **Tech-support**: apakah gender berhubungan dengan penghasilan > $50K?


In [ ]:
small = adult[adult.occupation == 'Tech-support'].sample(30, random_state=6)
tab = pd.crosstab(small['sex'], small['income'])
print(tab)

chi_y = stats.chi2_contingency(tab, correction=True)
chi_n = stats.chi2_contingency(tab, correction=False)
fisher = stats.fisher_exact(tab.values)
print(f"\nExpected:\n{np.round(chi_y[3], 2)}")
print(f"Expected minimum = {chi_y[3].min():.2f}  → < 5, syarat chi-square dilanggar")
print(f"\nChi-square tanpa koreksi : p = {chi_n[1]:.3f}")
print(f"Chi-square + Yates       : p = {chi_y[1]:.3f}")
print(f"Fisher exact             : p = {fisher.pvalue:.3f}  ← pilihan yang tepat")
print("\n💡 Tiga metode, tiga p-value — dan kesimpulannya berbeda di sekitar α = 0.05.")
print("   Untuk tabel 2×2 dengan expected < 5, laporkan Fisher exact.")


## H.3 Ringkasan Semua Uji


In [ ]:
ringkas = pd.DataFrame([
    ['Jam kerja pria vs wanita', 'Welch t-test', t_welch.pvalue, f"d = {cohens_d(male, female):.2f}"],
    ['Capital gain > $50K vs ≤ $50K', 'Mann-Whitney U', mw.pvalue, f"P(X>Y) = {ps:.2f}"],
    ['Nadi jalan kaki menit 1 vs 15', 'Paired t-test', paired.pvalue, f"d̄ = {walk.d.mean():.1f} bpm"],
    ['  (desain salah: independen)', 'Welch t-test', indep.pvalue, '—'],
    ['Proporsi > $50K pria vs wanita', 'z dua proporsi', p_prop, f"RR = {pm/pf:.2f}"],
    ['Pendidikan × penghasilan', 'Chi-square', p_chi, f"V = {cramers_v(ct):.2f}"],
    ['Gender × penghasilan (n = 30)', 'Fisher exact', fisher.pvalue, '—'],
], columns=['Perbandingan', 'Uji', 'p-value', 'Effect size'])
ringkas['Keputusan'] = np.where(ringkas['p-value'] < ALPHA, 'Tolak H₀', 'Gagal tolak H₀')
ringkas['p-value'] = ringkas['p-value'].apply(lambda v: f"{v:.2e}" if v < 0.001 else f"{v:.4f}")
ringkas


---

# 📝 I. Template Milestone 5 + Refleksi

## I.1 Template untuk Dataset Projek Anda


In [ ]:
# =========================================================
# TEMPLATE MILESTONE 5 — ganti dengan dataset & kolom Anda
# =========================================================
# df_my = pd.read_csv("<URL dataset Milestone 1-4 Anda>")

# ---------- 1. NUMERIK × 2 GRUP ----------
# a = df_my.loc[df_my['<grup>'] == '<A>', '<numerik>'].dropna()
# b = df_my.loc[df_my['<grup>'] == '<B>', '<numerik>'].dropna()
# print(stats.skew(a), stats.skew(b), stats.levene(a, b))
# print(stats.ttest_ind(a, b, equal_var=False), welch_ci(a, b), cohens_d(a, b))
# Jika sangat skewed: print(stats.mannwhitneyu(a, b), prob_superiority(a, b))

# ---------- 2. KATEGORIK × KATEGORIK ----------
# tab = pd.crosstab(df_my['<kategori 1>'], df_my['<kategori 2>'])
# chi2, p, dof, exp = stats.chi2_contingency(tab)
# print(exp.min(), p, cramers_v(tab))
# sns.heatmap((tab - exp) / np.sqrt(exp), annot=True, cmap='RdBu_r', center=0)
# Jika 2×2 dan expected < 5: stats.fisher_exact(tab.values)

# ---------- 3. DUA PROPORSI atau BERPASANGAN ----------
# proportions_ztest([k1, k2], [n1, n2])   ATAU   stats.ttest_rel(after, before)

# ---------- 4. DISKUSI ----------
# Markdown: asosiasi vs kausalitas, kemungkinan confounder, keterbatasan data

print("💡 Copy template ini ke notebook projek Anda. Submit sebelum pertemuan Modul 14.")


## I.2 Pertanyaan Refleksi

1. **Welch vs pooled**: Mengapa Welch t-test disarankan sebagai default? Dalam kondisi apa pooled t-test bisa menyesatkan?

2. **n besar**: Semua perbandingan di Bagian C.1 menghasilkan p ≈ 0. Lalu bagaimana Anda menentukan variabel mana yang paling membedakan kelompok berpenghasilan tinggi?

3. **Paired**: Data jalan kaki memberi p = 0.007 dengan paired t-test, tetapi p = 0.27 jika dianggap independen. Jelaskan dengan bahasa sendiri mengapa hal ini terjadi. Berikan satu contoh data berpasangan dari dunia IT.

4. **Kausalitas**: Seorang teman menyimpulkan dari Bagian F, "Gender menyebabkan perbedaan penghasilan." Apa yang kurang tepat dari kesimpulan ini, dan apa yang ditunjukkan oleh tabel stratifikasi di F.1?

5. **Fisher**: Pada sampel 30 pekerja Tech-support, mengapa ketiga p-value berbeda? Mana yang Anda laporkan dan mengapa?

### ✏️ Jawaban Anda:

**1.** Welch t-test disarankan sebagai default karena tidak mengasumsikan varians kedua kelompok harus sama (equal variances). Welch t-test tetap andal (robust) baik ketika varians kedua kelompok sama maupun berbeda, serta saat ukuran sampel tidak seimbang. Sebaliknya, pooled t-test berasumsi varians kedua kelompok sama; jika asumsi ini dilanggar (terutama pada ukuran sampel yang tidak seimbang), pooled t-test bisa menyesatkan dengan menghasilkan tingkat kesalahan Tipe I (false positive) yang salah atau nilai p-value yang tidak akurat.

**2.** Pada ukuran sampel yang sangat besar ($n$ besar), p-value hampir selalu mendekati 0 karena uji statistik menjadi sangat sensitif bahkan terhadap perbedaan terkecil sekalipun. Untuk menentukan variabel mana yang paling membedakan, kita tidak bisa mengandalkan p-value, melainkan harus melihat **Effect Size (ukuran efek)** seperti **Cohen's d**. Variabel dengan nilai absolut Cohen's d terbesar adalah variabel yang paling kuat membedakan kedua kelompok (memiliki efek praktis terbesar).

**3.**
- **Penjelasan**: Uji independen mencampur variabilitas (noise) *antar-individu* dengan variabilitas *efek perlakuan*. Karena variasi denyut nadi dasar antar-orang sangat besar (ada yang bawaannya rendah, ada yang tinggi), perbedaan kecil akibat jalan kaki (+3.5 bpm) menjadi tenggelam dalam noise tersebut. Sedangkan pada *paired t-test*, kita mengeliminasi variabilitas antar-orang dengan langsung menghitung selisih (d) sebelum dan sesudah pada orang yang sama, sehingga mengisolasi efek murni dari aktivitas.
- **Contoh di dunia IT**: Pengujian performa sistem (latency) sebelum dan sesudah optimasi query database pada 50 endpoint API yang sama (pengukuran berulang/sebelum-sesudah pada subjek yang sama).

**4.** Kesimpulan tersebut kurang tepat karena data yang digunakan adalah **data observasional**, bukan eksperimen terkontrol (RCT), sehingga korelasi/asosiasi tidak serta-merta menunjukkan hubungan sebab-akibat (kausalitas). Tabel stratifikasi di F.1 menunjukkan bahwa ketika kita mengontrol variabel pengganggu (confounder) seperti tingkat pendidikan dan jam kerja (full-time vs part-time), selisih proporsi tersebut berubah. Ini membuktikan bahwa perbedaan penghasilan dipengaruhi oleh banyak faktor lain yang saling terkait (seperti jam kerja dan tingkat pendidikan), bukan semata-mata disebabkan oleh gender secara langsung.

**5.** Ketiga p-value berbeda karena masing-masing metode menangani ukuran sampel kecil dengan cara yang berbeda:
- **Chi-square tanpa koreksi**: Cenderung terlalu optimis (p-value terlalu kecil) ketika ukuran sampel kecil karena mengasumsikan distribusi kontinu pada data diskrit.
- **Chi-square + Yates**: Memberikan koreksi kontinuitas yang cenderung terlalu konservatif (p-value terlalu besar).
- **Fisher exact**: Menghitung probabilitas eksak secara kombinatorial tanpa asumsi distribusi sampel besar.

**Yang dilaporkan**: Kita harus melaporkan **Fisher exact** karena syarat uji Chi-square dilanggar (terdapat sel pada expected table yang nilainya < 5, yaitu hanya 1.70 dan 4.30). Fisher exact adalah metode paling akurat dan tepat untuk tabel kontingensi 2x2 dengan sampel kecil.

## I.3 Checklist Submit

### Praktikum (notebook ini)
- [ ] Semua cell berjalan tanpa error
- [ ] Welch vs pooled + Levene + CI + Cohen's d
- [ ] Tabel "n besar" dengan Cohen's d per variabel
- [ ] Mann-Whitney + P(X > Y) + kurva survival capital gain
- [ ] Paired t-test vs uji independen + plot garis per orang
- [ ] Uji dua proporsi + RR + tabel stratifikasi
- [ ] Chi-square independensi + heatmap residual + Cramér's V
- [ ] Goodness-of-fit (acak vs bias) + Fisher exact
- [ ] 5 pertanyaan refleksi dijawab

### Milestone 5 (notebook projek Anda)
- [ ] Minimal 1 perbandingan numerik 2 grup (t / Mann-Whitney)
- [ ] Minimal 1 chi-square independensi + residual + Cramér's V
- [ ] Uji dua proporsi ATAU uji berpasangan (jika datanya ada)
- [ ] Cek asumsi; Fisher / non-parametrik bila dilanggar
- [ ] Effect size + CI untuk setiap perbandingan
- [ ] Diskusi asosiasi vs kausalitas + kemungkinan confounder
- [ ] README merangkum seluruh temuan Milestone 1-5
- [ ] Commit: `"Milestone 5: two-sample tests & chi-square"`, link via LMS **sebelum Modul 14**

---

## 🎓 Selamat! Anda Berhasil...

✅ Memilih uji berdasarkan desain data (independen vs berpasangan)
✅ Melaporkan effect size, bukan hanya p-value
✅ Membuktikan kekuatan desain berpasangan
✅ Membaca residual chi-square dan memakai Fisher exact saat sel kecil
✅ Membedakan asosiasi dari sebab-akibat

## 📚 Bacaan Lanjutan
- **Bruce, P.** (2020). *Practical Statistics for Data Scientists*, Ch. 3 (t-Tests, Chi-Square Test, Fisher's Exact Test).
- **Kohavi, R. & Becker, B.** (1996). Adult Data Set. UCI Machine Learning Repository.
- **Delacre, M., Lakens, D. & Leys, C.** (2017). *Why Psychologists Should by Default Use Welch's t-test Instead of Student's t-test*. International Review of Social Psychology.

---

*Notebook by: Prodi Teknik Informatika · Uniska MAB Banjarmasin · Edisi 2026*
